# Co-NON-response receptor graph (architecture D) — results

**Compute runs externally**; this notebook only loads results and plots them.

Twin of architecture A (co-response), but the receptor graph is built from **tested NEGATIVES**: two
receptors are neighbours if the SAME molecules FAIL to activate both (cosine over `S_neg` columns).
All three receptor graphs run side by side — only the smoothing neighbour source changes:

- **raw** — `[mol | prot]`.
- **nbr** — + molecule (chemical) neighbour CF at the target protein.
- **nbr_esm** — nbr + smoothed over **ESM** receptor neighbours (x / sequence).
- **nbr_cor** — nbr + smoothed over **co-response** receptor neighbours (y / signed; = arch A).
- **nbr_nor** — nbr + smoothed over **co-non-response** receptor neighbours (y / negatives; = arch D).

Both regimes, **5 repeats**. Inductive cold split **stratified**. Binary weights. Embeddings standardised.

### Run
```bash
MAX_PARALLEL=10 bash scripts/modeling/eval/run_noresponse_graph.sh
```

In [ ]:
import sys, pathlib, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import t as student_t

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('run from inside the orbind repository')
    ROOT = ROOT.parent

RESULT_DIR = ROOT / 'results/graph/noresponse'
COMBINED = RESULT_DIR / 'tables/noresponse_runs.csv'
RUNS_DIR = RESULT_DIR / 'tables/runs'
FIGURES_DIR = ROOT / 'notebooks/figures'; FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS = ['AUROC', 'AUPRC', 'MCC', 'F1']
REGIMES = ['transductive', 'inductive_molecule']

if COMBINED.exists():
    RES = pd.read_csv(COMBINED)
    print('loaded', COMBINED.relative_to(ROOT))
elif RUNS_DIR.exists() and glob.glob(str(RUNS_DIR / '*.csv')):
    RES = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(RUNS_DIR / '*.csv')))], ignore_index=True)
    print(f'loaded {RES.repeat.nunique()} repeat-files from', RUNS_DIR.relative_to(ROOT))
else:
    RES = pd.DataFrame()
    print('No results yet.')

def ci95_half(x):
    x = pd.Series(x).dropna().astype(float)
    if len(x) < 2:
        return np.nan
    return float(student_t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)))

if len(RES):
    print('repeats per regime:', RES.groupby('regime').repeat.nunique().to_dict())
    print('mean test prevalence:', RES.groupby('regime').test_prevalence.mean().round(3).to_dict())
    diag = RES.groupby('regime')[['coresp_sim','nonresp_sim','jaccard_nor_esm','jaccard_nor_cor']].mean().round(3)
    print('\nreceptor-graph diagnostics (similarities and how distinct the non-response graph is):')
    display(diag)
    display(RES.groupby(['regime', 'bar'])[METRICS].mean().round(3))

In [ ]:
# ---- bar-grid: regimes (rows) x metrics (cols), mean +/- 95% t-CI over repeats ----
ORDER = ['raw', 'nbr', 'nbr_esm', 'nbr_cor', 'nbr_nor']
LABEL = {'raw':'raw','nbr':'nbr','nbr_esm':'nbr\\n+ESM','nbr_cor':'nbr\\n+co-resp','nbr_nor':'nbr\\n+non-resp'}
BAR_C = {'raw':'#9a9a95','nbr':'#c8c8c2','nbr_esm':'#2a78d6','nbr_cor':'#e07b39','nbr_nor':'#b0439b'}

if not len(RES):
    print('No results to plot yet.')
else:
    fig, axes = plt.subplots(len(REGIMES), len(METRICS), figsize=(15, 7), squeeze=False)
    for ri, regime in enumerate(REGIMES):
        d = RES[RES.regime == regime]
        bars = [b for b in ORDER if b in set(d.bar)]
        for mi, metric in enumerate(METRICS):
            ax = axes[ri, mi]
            means = [d[d.bar == b][metric].mean() for b in bars]
            errs = [ci95_half(d[d.bar == b][metric]) for b in bars]
            x = np.arange(len(bars))
            ax.bar(x, means, color=[BAR_C[b] for b in bars], edgecolor='white', linewidth=1.2,
                   yerr=np.nan_to_num(errs), error_kw=dict(ecolor='#33322f', capsize=3, lw=1), zorder=3)
            if means:
                ax.axhline(means[0], color=BAR_C[bars[0]], ls='--', lw=1, zorder=1)
            ax.set_ylim(0, 1); ax.grid(alpha=.15, axis='y')
            ax.set_xticks(x); ax.set_xticklabels([LABEL[b] for b in bars], fontsize=8)
            for s in ('top', 'right'): ax.spines[s].set_visible(False)
            if ri == 0: ax.set_title(metric, fontsize=11)
            for xi, v in zip(x, means):
                if np.isfinite(v):
                    ax.text(xi, v + .02, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
        nrep = d.repeat.nunique()
        axes[ri, 0].set_ylabel(f'{regime}\n({nrep} repeats)', fontsize=10)
    fig.suptitle('Receptor graph source: sequence (ESM) vs co-response vs co-non-response — mean +/- 95% t-CI', fontsize=12, y=1.0)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'noresponse_graph.png', dpi=150, bbox_inches='tight')
    plt.show()

## How to read

- The new bar is **nbr_nor (co-non-response, y from negatives)** vs **nbr_cor (co-response)** vs
  **nbr_esm (sequence)** — does agreement on what receptors *reject* carry signal beyond co-response and ESM?
- The diagnostics (`nonresp_sim`, `jaccard_nor_esm`, `jaccard_nor_cor`) show how distinct the non-response
  graph is from the other two; low Jaccard ⇒ it's genuinely a different receptor neighbourhood.
- Compare on **AUROC** primarily. Whiskers = 95% t-CI over 5 repeats. Given the ceiling story, expect
  small deltas — the read is which y-structure (response vs non-response vs sequence) carries most.

Knobs: `K`, `KP`, `MAX_PARALLEL`. Delete `results/graph/noresponse/tables/` to regen.